In [ ]:
# 1. Import the module we need to patch
import scipy.integrate

# 2. Check if the patch is needed to avoid errors
if not hasattr(scipy.integrate, 'simps'):
    print("Monkey patching scipy.integrate: 'simps' not found. Pointing to 'simpson'.")
    # 3. Create the 'simps' attribute and point it to the existing 'simpson' function.
    scipy.integrate.simps = scipy.integrate.simpson
else:
    print("'simps' already exists in scipy.integrate. No patch needed.")

import strawberryfields as sf
from strawberryfields import ops
from strawberryfields.ops import Sgate, BSgate, MeasureFock

from strawberryfields.tdm import shift_by
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm

import strawberryfields as sf
from strawberryfields.ops import *
import numpy as np
from scipy.linalg import sqrtm

# import warnings
# from scipy.linalg import LinAlgWarning
# warnings.simplefilter('always', LinAlgWarning)  # show every occurrence

# fidelity

In [ ]:
def simple_fidelity(rho, sigma):
    """Calculates the Uhlmann-Jozsa fidelity between two density matrices."""
    
    # Ensure inputs are numpy arrays
    rho = np.array(rho)
    sigma = np.array(sigma)
    
    # Calculate the square root of rho
    # sqrtm is the matrix square root, not element-wise
    rho_sqrt = sqrtm(rho)
    
    # Calculate the product inside the trace
    product = rho_sqrt @ sigma @ rho_sqrt
    
    # Calculate the square root of the product
    sqrt_product = sqrtm(product)
    
    # The trace of the result is the "trace distance" part
    # We take the real part to handle potential small imaginary numerical errors
    trace = np.trace(sqrt_product).real
    
    # Fidelity is the square of this trace
    fidelity = trace**2
    
    return fidelity

def robust_fidelity(rho, sigma):
    """
    Calculates the Uhlmann-Jozsa fidelity with enhanced numerical robustness.

    This implementation explicitly uses eigendecomposition and includes steps to
    handle common floating-point precision issues.
    """
    # --- 1. Input Validation and Conditioning ---
    rho = np.asarray(rho, dtype=np.complex128)
    sigma = np.asarray(sigma, dtype=np.complex128)
    
    if rho.shape != sigma.shape or rho.ndim != 2 or rho.shape[0] != rho.shape[1]:
        raise ValueError("Input density matrices must be square and have the same shape.")

    # Enforce Hermiticity on inputs to remove numerical noise
    rho = 0.5 * (rho + rho.T.conj())
    sigma = 0.5 * (sigma + sigma.T.conj())

    # --- 2. Calculate sqrt(rho) Robustly ---
    # eigh is best for Hermitian matrices
    e_vals_rho, e_vecs_rho = np.linalg.eigh(rho)
    
    # Clip small negative eigenvalues to 0 due to numerical instability
    e_vals_rho_clipped = np.maximum(e_vals_rho.real, 0)
    
    # Calculate square root of eigenvalues
    sqrt_e_vals_rho = np.sqrt(e_vals_rho_clipped)
    
    # Reconstruct sqrt(rho) = U * sqrt(D) * U_dagger
    rho_sqrt = e_vecs_rho @ np.diag(sqrt_e_vals_rho) @ e_vecs_rho.T.conj()
    
    # --- 3. Calculate the product matrix K and ensure it's Hermitian ---
    K = rho_sqrt @ sigma @ rho_sqrt
    K = 0.5 * (K + K.T.conj()) # Enforce Hermiticity on the result

    # --- 4. Calculate Tr(sqrt(K)) Robustly ---
    # We only need the eigenvalues of K. Use eigvalsh for efficiency.
    e_vals_K = np.linalg.eigvalsh(K)
    
    # Clip again before the final square root
    e_vals_K_clipped = np.maximum(e_vals_K.real, 0)
    
    # The trace of sqrt(K) is the sum of the square roots of K's eigenvalues
    trace_val = np.sum(np.sqrt(e_vals_K_clipped))
    
    # --- 5. Calculate and Clip Final Fidelity ---
    fidelity = trace_val**2
    
    # Clip the final result to the valid [0, 1] range
    return np.clip(fidelity, 0.0, 1.0)

# reward states

In [ ]:
cutoff = 20
eng = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})

prog = sf.Program(1)

targe_dm = []
# Define the parameters for the squeezed cat state
alpha = 3
r = 1.38

# --- 2. Construct the Quantum Circuit ---
with prog.context as q:
    # First, prepare the initial cat state
    # A cat state is a superposition of two coherent states
    Catstate(alpha) | q[0]
    
    # Second, apply the squeezing operation
    Sgate(r) | q[0]

state_theoretical_dm_rho_plus = eng.run(prog).state.dm(cutoff=cutoff)
targe_dm.append(state_theoretical_dm_rho_plus)

prog = sf.Program(1)
with prog.context as q:
    # First, prepare the initial cat state
    # A cat state is a superposition of two coherent states
    Catstate(alpha,p=1) | q[0]
    
    # Second, apply the squeezing operation
    Sgate(r) | q[0]

state_theoretical_dm_rho_minus = eng.run(prog).state.dm(cutoff=cutoff)
targe_dm.append(state_theoretical_dm_rho_minus)

prog = sf.Program(1)
with prog.context as q:
    # First, prepare the initial cat state
    # A cat state is a superposition of two coherent states
    Catstate(alpha) | q[0]
    
    # Second, apply the squeezing operation
    Sgate(r) | q[0]

    Rgate(np.pi/2) | q[0]

state_theoretical_dm_rho_plus_rot = eng.run(prog).state.dm(cutoff=cutoff)
targe_dm.append(state_theoretical_dm_rho_plus_rot)

prog = sf.Program(1)
with prog.context as q:
    # First, prepare the initial cat state
    # A cat state is a superposition of two coherent states
    Catstate(alpha,p=1) | q[0]
    
    # Second, apply the squeezing operation
    Sgate(r) | q[0]

    Rgate(np.pi/2) | q[0]

state_theoretical_dm_rho_minus_rot = eng.run(prog).state.dm(cutoff=cutoff)
targe_dm.append(state_theoretical_dm_rho_minus_rot)

# circuit

In [ ]:
# Parameters
r0 = 1.38  # Squeezing parameter
tau_1 = 0.367
tau_2 = 0  # Switchable mirror for VBS2

# # Convert transmission coefficient to beam splitter angle.
theta_2 = np.arccos(tau_2)

# Create a Strawberry Fields program
eng = sf.Engine("fock", backend_options={"cutoff_dim": 20})
results = []

index=1
while True:
    # agent can control tau1
    theta_1 = np.arccos(tau_1)

    # Create a new program for each iteration. it must be set before each iteration
    # note that the variable q[1] and q[2] are not changed when setting the program
    prog = sf.Program(2)

    with prog.context as q:
        # MeasureFock() | q[0]  # Measure the first mode
        # MeasureFock() | q[1]  # Measure the second mode
        # Initialize mode 0 with a squeezed vacuum state
        # agent can control squeezed state rotation, but not the squeezing strength
        rot_theta=index
        Sgate(1.38, 0) | q[1]

        # Apply variable beam splitter (VBS1)
        BSgate(theta_1, 0) | (q[0], q[1])

        # Photon-number-resolving measurement (PNR)
        MeasureFock(select=4) | q[0]

        BSgate(theta_2, 0) | (q[0], q[1])  # Fully reflective mirror

    # Run the simulation and store result
    result = eng.run(prog)
    state_dm=result.state.reduced_dm(modes=[0])
    fidelities = [robust_fidelity(sigma=state_dm, rho=target) for target in targe_dm]
    max_fidelity = np.max(fidelities) if len(fidelities) > 0 else 0.0
    reward = max_fidelity ** 50
    fid1 = robust_fidelity(state_dm,state_theoretical_dm_rho_plus)
    fid2 = simple_fidelity(state_dm,state_theoretical_dm_rho_plus)
    assert(np.isclose(fid1,fid2))
    print(reward)
    index = index+1
    if index>5:
        break
    

# Use the last result for plotting
print("Final state:", result.state)
print(result.samples)

xvec = np.linspace(-15, 15, 401)                        # Create vector of 401 points from -15 to 15 for phase-space coordinates
W = result.state.wigner(mode=0, xvec=xvec, pvec=xvec)   # Calculate Wigner function for quantum state (mode 1)
scale = np.max(W.real)                                  # Get max value for symmetric color scaling
nrm = mpl.colors.Normalize(-scale, scale)               # Create symmetric color normalization
plt.axes().set_aspect("equal")                          # Set square aspect ratio for phase-space plot
plt.contourf(xvec, xvec, W, 60, cmap=cm.RdBu, norm=nrm) # Plot Wigner function using filled contours (red: positive, blue: negative)
plt.show()